# 黑马头条项目｜第06天：Kafka及异步通知文章上下架

> 学习目标：理解 Kafka 的消息模型、高可用与消费语义；在自媒体服务修改文章上下架状态后，通过消息通知文章服务更新 `ap_article_config`，让两个服务保持最终一致。

本篇依据百度网盘 `day06-kafka及异步通知文章上下架/讲义/kafka及异步通知文章上下架.md` 的网页预览整理。未启动 ZooKeeper、Kafka、Docker、Nacos、MySQL 或课程微服务，未执行接口和消息测试；课程配置中的固定地址已替换为占位符。代码是讲义关键逻辑的学习摘录，部分工程化注意点为补充。


## 1. 业务需求：文章上下架

自媒体端管理已经发布的文章：

- 已发表且已上架的文章可以下架；
- 已发表且已下架的文章可以重新上架；
- 草稿、审核中或审核失败的文章不能直接执行上下架。

文章的管理状态和 APP 端的展示配置分别由不同服务维护。自媒体服务拥有 `wm_news.enable`，文章服务拥有 `ap_article_config.is_down`。如果只更新前者，APP 端仍可能展示旧状态，因此需要异步通知：

```text
客户端
  │ POST /api/v1/news/down_or_up
  ▼
wemedia：校验文章、更新 wm_news.enable
  │ Kafka topic: wm.news.up.or.down.topic
  ▼
article：消费消息、更新 ap_article_config.is_down
```

这是最终一致性流程。接口成功表示自媒体端已接受状态变更，不代表文章端已经完成消费；前端查询、重试和运营提示要接受这段短暂延迟。


## 2. 消息中间件选型与 Kafka 名词

讲义对比 ActiveMQ、RabbitMQ、RocketMQ 和 Kafka：Kafka侧重高吞吐量和分布式日志，RabbitMQ功能较完整，RocketMQ在可靠性和扩展性方面常用于大型业务。具体吞吐量会受到版本、硬件、消息大小、分区数和确认策略影响，表格中的“万级/百万级”只能作为课程中的概念比较。

Kafka 是分布式流式平台，也可作为消息队列使用。核心名词：

| 名词 | 含义 |
|---|---|
| Producer | 发布消息的生产者 |
| Topic | 消息分类的主题 |
| Partition | Topic 内的有序分区，消息以 offset 标识位置 |
| Consumer | 订阅并处理消息的消费者 |
| Consumer Group | 一组协作消费的消费者实例 |
| Broker | Kafka 集群中的服务器节点 |
| Offset | 某个分区内消息的位置，消费者用它恢复进度 |

同一个 Topic 中，**同一消费者组内一个分区在同一时刻只分配给一个消费者**；不同消费者组会各自收到一份消息。这就同时支持队列模型和发布-订阅模型。


## 3. Topic、分区与两种消费关系

```text
Producer ──► Topic ──► Consumer Group A（实例A1、A2）
                    └► Consumer Group B（实例B1）
```

- Group A 内，消息按分区分摊给 A1/A2，一条消息只由其中一个实例处理；适合横向扩展消费者。
- Group B 是另一组，仍会读取这条消息；适合文章服务、审计服务各自订阅同一事件。
- 一个消费者组的实例数超过分区数时，多出的实例暂时没有分区可消费。
- 同一分区内消息按 offset 追加，消费者通常按顺序拉取；跨分区没有全局顺序。

文章上下架事件通常使用一个独立 Topic，article 服务使用自己的消费者组。若要保证同一文章的上架和下架事件顺序，生产消息时可以将 `articleId` 作为 key，使同一文章路由到同一分区；仍需在消费端按版本或时间校验，避免迟到事件覆盖新状态。


## 4. Kafka 高可用：Broker、Leader、Follower 与 ISR

Kafka 集群由多个 Broker 组成。Topic 的每个分区可以有多个副本：

- **Leader Replica**：负责该分区的读写入口；
- **Follower Replica**：从 Leader 同步数据；
- **ISR（in-sync replicas）**：与 Leader 保持同步、具备参与选举资格的副本集合。

当 Leader 所在 Broker 宕机，Kafka 优先从 ISR 中选择新的 Leader。若 ISR 都不可用，可以等待可靠副本恢复，或选择其他副本尽快恢复可用性；后者可能存在数据不完整风险。副本数、最小同步副本数和生产者 `acks` 应结合可靠性与吞吐量设计。

课程资料使用 ZooKeeper 保存 Kafka 的集群元数据，并给出 Docker 安装示例。新版本 Kafka 可采用 KRaft 模式；本笔记不执行课程中的镜像拉取和容器命令，实际部署应按目标 Kafka 版本选择架构。


## 5. 原生 Java 客户端：生产者和消费者

### 5.1 生产者配置骨架

```java
Properties props = new Properties();
props.put(ProducerConfig.BOOTSTRAP_SERVERS_CONFIG,
          "${KAFKA_BOOTSTRAP_SERVERS}");
props.put(ProducerConfig.RETRIES_CONFIG, 5);
props.put(ProducerConfig.KEY_SERIALIZER_CLASS_CONFIG,
          StringSerializer.class.getName());
props.put(ProducerConfig.VALUE_SERIALIZER_CLASS_CONFIG,
          StringSerializer.class.getName());

try (KafkaProducer<String, String> producer =
         new KafkaProducer<>(props)) {
    ProducerRecord<String, String> record =
        new ProducerRecord<>("demo-topic", "article-1", "hello kafka");
    producer.send(record);
}
```

生产者关闭前应等待或正确处理发送结果；`send` 的异步调用只表示请求已提交到客户端缓冲区，不等于 Broker 已持久化。

### 5.2 消费者轮询

```java
props.put(ConsumerConfig.GROUP_ID_CONFIG, "article-service");
props.put(ConsumerConfig.KEY_DESERIALIZER_CLASS_CONFIG,
          StringDeserializer.class.getName());
props.put(ConsumerConfig.VALUE_DESERIALIZER_CLASS_CONFIG,
          StringDeserializer.class.getName());

consumer.subscribe(Collections.singletonList("demo-topic"));
while (running) {
    ConsumerRecords<String, String> records =
        consumer.poll(Duration.ofMillis(1000));
    for (ConsumerRecord<String, String> record : records) {
        handle(record.key(), record.value());
    }
}
```

真实应用还要在 finally 中关闭 consumer，设置拉取、处理和再均衡相关参数，并对处理异常建立重试或死信策略。


## 6. 生产者发送方式与关键参数

### 同步和异步发送

同步发送等待 Future 结果，能立即知道发送是否成功，但会降低吞吐；异步发送配合回调，适合高吞吐场景，失败要写日志、重试或进入补偿表。

```java
// 同步：只在确实需要等待结果时使用
RecordMetadata metadata = producer.send(record).get();

// 异步：处理完成回调中的异常
producer.send(record, (metadata, exception) -> {
    if (exception != null) {
        log.error("Kafka send failed", exception);
    }
});
```

### `acks`、重试和压缩

| 参数 | 作用 | 取舍 |
|---|---|---|
| `acks=0` | 不等待 Broker 响应 | 快，但丢失风险高 |
| `acks=1` | Leader 写入后确认 | 延迟较低，Leader故障时仍有窗口 |
| `acks=all` | ISR 满足确认条件后返回 | 可靠性更高，延迟和资源开销更大 |
| `retries` | 临时错误的重试次数 | 需结合超时、幂等和顺序要求 |
| `compression.type` | 压缩消息 | 节省网络/存储，增加 CPU |

讲义列出 `snappy`、`lz4`、`gzip`；生产环境选择要以压测和消息大小为依据。只增加 `retries` 不能解决业务重复，应用侧仍要用事件ID或业务版本幂等。

**顺序的前提：** 同一分区按序拉取不保证业务异步线程按序完成。要保持最终状态正确，还应限制同一业务键的处理并发。生产者分区数改变、分区策略改变或多个生产者并行发送时，单靠相同key也不足以定义业务先后顺序。


## 7. 消费者组、顺序和再均衡

消费者组通过分区分配实现并行消费。消费者加入、退出、崩溃或分区数变化时会触发再均衡，分区被重新分配给存活的消费者。

顺序规则：

1. 单个分区内，Kafka按 offset 提供有序读取；
2. 一个消费者组内，一个分区同一时刻只由一个实例处理；
3. 多个分区之间没有全局顺序；
4. 想让某个业务键有序，应让相同 key 进入同一分区，并控制消费并发。

上下架事件如果被拆到多个分区，文章A和文章B之间不需要互相排序；同一文章的 enable=0、enable=1 则应尽量固定分区，并在消息中携带版本或更新时间。即使固定分区，也要防止重试、网络延迟或旧消息重新投递带来的状态回退。

### 讲义提交示例的修正

讲义在遍历每条record时调用无参数 `commitSync()`。无参数提交可能覆盖本次poll已返回的全部分区进度，而后面的记录尚未处理，会形成“提前提交”窗口。上方整理示例将提交放在**整批记录全部处理成功之后**；若要逐条或局部提交，必须明确每个分区已经连续成功处理到的位置。

```java
// 只在这条记录及它之前同分区的记录均成功后，提交下一条待读位置
TopicPartition partition =
        new TopicPartition(record.topic(), record.partition());
consumer.commitSync(Collections.singletonMap(
        partition, new OffsetAndMetadata(record.offset() + 1)));
```

已提交offset表达的是**下一条要读取的位置**，不是最后一条成功记录本身；内部主题的正确名称是 `__consumer_offsets`。讲义中的名称存在拼写差异。

处理一批记录中途失败时，不应在finally里无条件提交整批已拉取进度。若采用“平时异步、退出同步”的策略，最后同步提交必须只包含确定处理成功的连续进度。单次commitSync也不是无期限重试直到必定成功，会因超时、再均衡等原因抛异常。


## 8. Offset 提交：自动、同步和异步

Kafka不是把“处理成功回执”直接写回队列，而是由消费者提交每个分区的 offset。消费者重启或再均衡后，从已提交位置继续读取。

### 自动提交

`enable.auto.commit=true` 时，客户端按周期提交最近一次 poll 的进度。若消息尚未处理完进程就崩溃，可能造成消息丢失；如果处理时间超过相关配置，也可能触发再均衡。

### 手动同步提交

关闭自动提交后，在业务处理成功后调用 `commitSync()`。同步提交会阻塞并在可恢复失败时重试，吞吐较低但语义更清楚。

```java
props.put(ConsumerConfig.ENABLE_AUTO_COMMIT_CONFIG, false);
for (ConsumerRecord<String, String> record : records) {
    handle(record);
}
consumer.commitSync();
```

### 手动异步提交

`commitAsync` 不阻塞，但失败通常不会自动重试；多个异步提交乱序时，旧提交重试可能覆盖新 offset。课程建议在正常处理时异步提交，在关闭或不可恢复异常时最后同步提交。

**提交时机决定至少一次/至多一次倾向：** 先提交再处理可能丢消息，处理后提交可能重复。文章上下架必须使消费逻辑幂等，例如重复写同一个 `articleId` 的目标状态不会造成额外副作用。

**Spring监听容器与原生客户端的边界：** 上面的消费者配置补充关闭自动提交，但没有指定手动AckMode。Spring Kafka监听器的提交时机还取决于容器的 `AckMode`、批量/单条监听和错误处理配置；不能认为仅设置 `enable-auto-commit=false` 就已经实现了手动确认。若采用手动确认，应明确配置模式，并在数据库事务成功之后确认。


## 9. Spring Boot 集成 Kafka

课程示例使用 `spring-kafka` 和 `kafka-clients`，生产者与消费者都使用字符串序列化器；对象消息先转为 JSON 字符串，接收后再反序列化。

```yaml
spring:
  kafka:
    bootstrap-servers: ${KAFKA_BOOTSTRAP_SERVERS}
    producer:
      retries: 10
      key-serializer: org.apache.kafka.common.serialization.StringSerializer
      value-serializer: org.apache.kafka.common.serialization.StringSerializer
    consumer:
      group-id: ${spring.application.name}
      enable-auto-commit: false
      key-deserializer: org.apache.kafka.common.serialization.StringDeserializer
      value-deserializer: org.apache.kafka.common.serialization.StringDeserializer
```

```java
@RestController
class MessageController {
    private final KafkaTemplate<String, String> kafkaTemplate;

    @GetMapping("/hello")
    String hello() {
        kafkaTemplate.send("demo-topic", "hello");
        return "ok";
    }
}

@Component
class MessageListener {
    @KafkaListener(topics = "demo-topic")
    void onMessage(String message) {
        if (StringUtils.hasText(message)) {
            // 处理消息；成功后再按策略提交 offset
        }
    }
}
```

对象消息示意：`kafkaTemplate.send("user-topic", JSON.toJSONString(user))`；监听器中用 `JSON.parseObject(message, User.class)`。生产上应约定 schema、版本和字段兼容规则，避免任意 JSON 被当成可信对象。

### 讲义参数校验的补充

讲义仅在 `enable` 非空且属于0/1时执行更新，但在无效值时仍返回成功。更合理的是在入口对 `dto`、`id`、`enable` 全部校验，不合法就返回参数错误；更新失败时也不能继续声称成功或发送状态事件。后台同时发生删除、审核状态变化时，应通过带归属和状态条件的更新以及结果检查避免竞争。


## 10. 上下架接口定义

接口：`POST /api/v1/news/down_or_up`

```java
@Data
public class WmNewsDto {
    private Integer id;
    /** 0 下架，1 上架 */
    private Short enable;
}
```

业务校验顺序：

1. `id` 不能为空；
2. 查询 `wm_news`，不存在则返回数据不存在；
3. 只有已发布文章允许上下架；
4. `enable` 只能是 0 或 1；
5. 更新 `wm_news.enable`；
6. 文章已生成 `articleId` 时，发布上下架事件。

```java
@PostMapping("/down_or_up")
public ResponseResult downOrUp(@RequestBody WmNewsDto dto) {
    return wmNewsService.downOrUp(dto);
}
```

讲义将上下架字段加入完整 `WmNewsDto`。接口层还应校验当前用户对文章的归属，避免仅凭文章ID修改他人内容；这些权限细节是工程补充，不代表讲义已实现。


## 11. 自媒体端：更新本地状态并发送事件

核心事件结构：

```java
public final class WmNewsMessageConstants {
    public static final String WM_NEWS_UP_OR_DOWN_TOPIC =
            "wm.news.up.or.down.topic";
}

if (wmNews.getArticleId() != null) {
    Map<String, Object> event = new HashMap<>();
    event.put("articleId", wmNews.getArticleId());
    event.put("enable", dto.getEnable());
    kafkaTemplate.send(
        WmNewsMessageConstants.WM_NEWS_UP_OR_DOWN_TOPIC,
        JSON.toJSONString(event));
}
```

业务顺序需要仔细处理：如果数据库更新成功但 Kafka 发送失败，两个服务会暂时不一致；如果消息发送成功但数据库事务回滚，也会产生无效事件。讲义示例没有提供跨数据库和 Kafka 的原子方案，生产可使用 Outbox/本地消息表、事务消息或可靠重试：

```text
更新 wm_news + 写 outbox（同一DB事务）
             ↓
可靠发布 outbox 事件到 Kafka
             ↓
article 消费并幂等更新配置
```

`send` 返回的 Future/回调应记录失败。重试必须携带事件ID，避免同一上下架操作产生无法识别的重复消息。

### 避免数字类型比较导致错误上架

讲义使用 `Object enable = map.get("enable")`，再调用 `enable.equals(1)`。该写法在enable为空时会抛异常；若反序列化或其他调用得到 `Short`/`Long`，与 `Integer(1)` 比较也可能返回false，误判为下架。上方代码先统一转换为Integer，只是简化示意；完整实现还必须在转换前检查非空和格式，并明确拒绝0/1之外的值，不能把任意其他值默认为上架。

JSON字符串在Kafka中不代表类型安全。业务消息可用明确的 `ArticleEnableEvent` DTO替代原始Map，限制字段、校验版本并保留异常路径。


## 12. 文章端：监听消息并更新展示配置

文章服务配置自己的消费者组，监听同一个 Topic：

```java
@Component
@Slf4j
public class ArticleIsDownListener {
    private final ApArticleConfigService service;

    @KafkaListener(topics = WmNewsMessageConstants.WM_NEWS_UP_OR_DOWN_TOPIC)
    public void onMessage(String message) {
        if (StringUtils.isNotBlank(message)) {
            Map<String, Object> event = JSON.parseObject(message, Map.class);
            service.updateByMap(event);
            log.info("article config changed, articleId={}",
                     event.get("articleId"));
        }
    }
}
```

配置更新规则：`enable=0` 对应 `is_down=true`，`enable=1` 对应 `is_down=false`。

```java
@Transactional
public void updateByMap(Map<String, Object> event) {
    Integer enable = Integer.valueOf(event.get("enable").toString());
    boolean isDown = enable == 0;
    update(Wrappers.<ApArticleConfig>lambdaUpdate()
        .eq(ApArticleConfig::getArticleId, event.get("articleId"))
        .set(ApArticleConfig::getIsDown, isDown));
}
```

消费前要校验 `articleId`、`enable`、事件版本和消息格式；更新影响行数为0时记录告警。监听器抛出异常时，应明确 Spring Kafka 的错误处理、重试和死信 Topic，不能静默丢弃。


## 13. 可靠性：重复、丢失和乱序如何处理

### 重复消费

消费者处理成功但提交 offset 失败，重启后会再次收到消息。更新同一行相同值通常是幂等的，但日志、通知、计费等副作用必须用事件ID去重。

### 消息丢失

- 生产者 `acks=0` 或未处理发送失败；
- 自动提交早于业务处理；
- 数据库提交后事件未可靠发布；
- 消费者提交 offset 后进程才开始业务处理。

### 乱序

上下架事件可能因多个分区、重试或旧事件延迟而乱序。生产者用 `articleId` 作为 key 固定分区，并在消息中携带 `eventId`、`version` 或 `updatedAt`；消费者只接受不早于当前状态版本的事件。

### 推荐事件字段

```json
{
  "eventId": "<UUID>",
  "articleId": 123,
  "enable": 1,
  "version": 7,
  "occurredAt": "<ISO-8601-TIME>"
}
```

示例中的 UUID 和时间为占位结构，不应把真实凭据放入消息。事件schema一旦被多个服务依赖，应建立兼容演进规则。


## 14. 配置与部署边界

课程通过公共模块引入 Kafka 依赖，并在 Nacos 配置中心分别配置生产者和消费者。配置示意：

```yaml
spring:
  kafka:
    bootstrap-servers: ${KAFKA_BOOTSTRAP_SERVERS}
    producer:
      retries: 10
      key-serializer: org.apache.kafka.common.serialization.StringSerializer
      value-serializer: org.apache.kafka.common.serialization.StringSerializer
    consumer:
      group-id: ${spring.application.name}
      key-deserializer: org.apache.kafka.common.serialization.StringDeserializer
      value-deserializer: org.apache.kafka.common.serialization.StringDeserializer
```

不同服务必须使用正确的消费者组：article 服务要独立于其他订阅者，否则同一事件会被组内实例分摊而不是广播给每个业务。Broker 地址、账号和密码应由环境变量或密钥管理注入，不能写入源码、笔记或日志。

课程使用 ZooKeeper + Kafka 容器；本篇没有执行任何 Docker 命令。部署前应确认 Kafka 版本、客户端版本、监听器地址、网络可达性、分区副本和数据保留策略。


## 15. 端到端流程与状态图

```text
用户请求上下架
      ↓
wemedia 校验 id/status/enable
      ↓
更新 wm_news.enable
      ↓
articleId 存在？ ──否──► 流程结束
      │是
      ▼
发送 wm.news.up.or.down.topic
      ↓
Kafka Broker 持久化并按 key 路由分区
      ↓
article 消费者组 poll
      ↓
解析 JSON + 校验版本
      ↓
更新 ap_article_config.is_down
      ↓
提交 offset / 失败重试或进入死信
```

状态映射：

| 自媒体 `enable` | 文章端 `is_down` | APP效果 |
|---:|---:|---|
| 0 | true | 下架，不对用户展示 |
| 1 | false | 上架，可以展示 |

第05天的“延迟任务”负责在指定时间触发文章审核；第06天的 Kafka 事件负责在文章已经存在后，把后续上下架操作通知给文章服务。两者解决的时间点不同，不能混写成同一条消息。


## 16. 验证清单（本次未运行环境）

| 场景 | 预期 | 检查点 |
|---|---|---|
| 已发布文章下架 | `wm_news.enable=0`，article配置最终为 `is_down=true` | 生产发送、Topic、消费日志 |
| 已下架文章上架 | `wm_news.enable=1`，article配置最终为 `is_down=false` | 消息字段和数据库行 |
| 草稿操作 | 返回参数错误，不发送事件 | 状态校验 |
| 不存在文章 | 返回数据不存在 | 查询与响应码 |
| 无 `articleId` | 仅更新自媒体，不发送 article 事件 | 分支条件 |
| Kafka暂时不可用 | 失败可观测并可补偿 | send回调、outbox/重试 |
| article重复消费 | 配置结果不变，副作用不重复 | eventId/版本幂等 |
| 消费者重启 | 从已提交offset继续，允许重复但不丢失 | commit时机 |
| 同文章快速上下架 | 最终状态以最新版本为准 | key分区、版本检查 |

排查链路：先看接口是否成功更新 `wm_news`，再看生产者发送结果、Broker 中 Topic/分区，再看 article 消费者组 offset 和监听日志，最后核对 `ap_article_config`。本篇未启动任何上述服务，因此没有运行结果可报告。


## 17. 复习与自测

### 核心结论

- 同一消费者组内消息按分区分摊；不同消费者组各自收到消息。
- Kafka只保证单分区顺序，跨分区没有全局顺序。
- `acks`、重试、压缩分别影响可靠性、失败恢复和传输成本。
- 消费者处理成功后再提交 offset，通常是至少一次语义，因此业务必须幂等。
- 自媒体更新 `wm_news.enable` 后发送事件，article 消费并更新 `ap_article_config.is_down`。
- 数据库更新与 Kafka 发送不是天然同一事务，可靠投递需要 Outbox、重试和监控。

### 自测题

1. **为什么 article 服务不能和其他消费者共用一个 group？** 共用后同一消息只会分配给组内一个实例，无法保证每个业务都收到；独立 group 才是发布-订阅。
2. **为什么同一个 articleId 要作为消息 key？** 让同一文章尽量进入同一分区，保留其事件顺序；仍应使用版本校验防止旧事件覆盖新状态。
3. **处理成功后提交 offset 仍会重复吗？** 会，提交失败或消费端崩溃都可能重新投递，所以更新和副作用要幂等。
4. **`acks=all` 是否代表业务绝不丢消息？** 只表示生产确认满足 Broker 的副本条件，数据库事务、网络、消费处理和最终业务仍需补偿。
5. **上下架接口返回成功是否代表 APP 已经完成下架？** 不代表，返回成功通常只覆盖自媒体端；Kafka消费存在异步延迟，需查询或监控最终状态。
6. **第05天的延迟队列与第06天 Kafka有什么区别？** 延迟队列控制文章审核任务何时可执行；Kafka传播上下架事件，解耦服务并支持异步通知。
